# Importing Libraries

In [1]:
import os

_ffmpeg_dlls = os.add_dll_directory(r"C:\Users\nova\miniconda3\envs\gpu\Library\bin")
import sys
import torch

print("Python executable:", sys.executable)
print("Environment:", sys.prefix)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python executable: C:\Users\nova\miniconda3\envs\gpu\python.exe
Environment: C:\Users\nova\miniconda3\envs\gpu
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti


In [2]:
from datasets import load_dataset, Audio
import torch
from transformers import pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
import pandas
from tqdm.auto import tqdm
from sklearn.metrics import classification_report
import soundfile
import io
import os
import numpy
from huggingface_hub import HfApi
import time
import jiwer
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score, classification_report
import json

W1117 15:23:26.079000 11396 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# Setup HERE

In [3]:
device = 0 if torch.cuda.is_available() else -1

folder_path = os.path.join( os.path.expanduser("~"), "Desktop", "Milestone2", "PipelineEvaluation" ) + os.sep
print(folder_path)

C:\Users\nova\Desktop\Milestone2\PipelineEvaluation\


# **Stage 2**

## Data splits

In [5]:
# Create version
api = HfApi()
dataset_info = api.dataset_info("DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb")
revision = dataset_info.sha
revision_filepath = f"{folder_path}dataset_revision.txt"
with open(revision_filepath, "w") as f:
    f.write(revision)
print("Dataset revision:", revision)

Dataset revision: 554ad4367e98b7c6f4d4d9756dc6bbdf345e042e


In [6]:
# Load exact dataset revision
"""
try:
    dataset = load_dataset("DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb", split="test", revision=revision)
    original_dataframe = pandas.DataFrame(dataset)
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()
"""
# If torchcodec gives a lot of trouble above
dataset = load_dataset("DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb", split="test", revision=revision)
dataset = dataset.cast_column("audio", Audio(decode=False))
original_dataframe = pandas.DataFrame(dataset)
print("Dataset loaded:", len(original_dataframe))

Dataset loaded: 3553


In [8]:
# Exclude samples labeled Disagreement & <mixed>
EXCLUDED_LABELS = ["Disagreement", "<mixed>"]
filtered_dataframe = original_dataframe[~original_dataframe["label"].isin(EXCLUDED_LABELS)].copy()
filtered_dataframe["original_index"] = filtered_dataframe.index

print("ORIGINAL dataset:")
print(original_dataframe["label"].value_counts())
print("\nFILTERED dataset:")
print(filtered_dataframe["label"].value_counts())

assert len(filtered_dataframe) == 3426
assert not filtered_dataframe["label"].isin(EXCLUDED_LABELS).any()

ORIGINAL dataset:
label
Neutral         2521
Positive         737
Negative         168
Disagreement     104
<mixed>           23
Name: count, dtype: int64

FILTERED dataset:
label
Neutral     2521
Positive     737
Negative     168
Name: count, dtype: int64


In [10]:
random_generator = numpy.random.default_rng(8001)
seed1 = int(random_generator.integers(0, 2**32))
seed2 = int(random_generator.integers(0, 2**32))

# First split: 500 development, 2,926 remaining
development_slice, remaining = train_test_split(filtered_dataframe,
    train_size=500, stratify=filtered_dataframe["label"],
    random_state=seed1)

# Second split: 1,500 test, 1,426 load pool
test_slice, load_pool = train_test_split(remaining,
    train_size=1500,stratify=remaining["label"],
    random_state=seed2
)

# Get original dataset indicies
development_indices = development_slice["original_index"].to_numpy()
test_indices = test_slice["original_index"].to_numpy()
load_pool_indices = load_pool["original_index"].to_numpy()

development_slice_size = len(development_slice)
test_slice_size = len(test_slice)
load_pool_size = len(load_pool)
total_size = development_slice_size + test_slice_size + load_pool_size
# Verify sizes
print(f"""Development slice size = {development_slice_size}
Test slice size = {test_slice_size}
Load pool size = {load_pool_size}
Total size = {total_size}
""")
assert development_slice_size == 500
assert test_slice_size == 1500
assert load_pool_size == 1426
assert total_size == 3426
print("All slice size checks passed!")

# Verify no overlapping indices
assert set(development_indices).isdisjoint(test_indices)
assert set(development_indices).isdisjoint(load_pool_indices)
assert set(test_indices).isdisjoint(load_pool_indices)
print("All overlap checks passed!")

# Save all  indices
numpy.save(f"{folder_path}development_indices.npy", numpy.array(development_indices))
numpy.save(f"{folder_path}test_indices.npy", numpy.array(test_indices))
numpy.save(f"{folder_path}load_pool_indices.npy", numpy.array(load_pool_indices))

# Save complete dataset slices
development_slice.to_csv(os.path.join(folder_path, "data", "development_slice.csv"), index=False)
test_slice.to_csv( os.path.join(folder_path, "data", "test_slice.csv"), index=False ) 
load_pool.to_csv( os.path.join(folder_path, "data", "load_pool_slice.csv"), index=False )

print("All dataset slices saved successfully!")

Development slice size = 500
Test slice size = 1500
Load pool size = 1426
Total size = 3426

All slice size checks passed!
All overlap checks passed!
All dataset slices saved successfully!


## For future data retrieval (run again)

In [8]:
# Get original dataset indicies
development_indices = numpy.load(f"{folder_path}/development_indices.npy")
test_indices = numpy.load(f"{folder_path}/test_indices.npy")
load_pool_indices = numpy.load(f"{folder_path}/load_pool_indices.npy")

# Reconstruct data slices
development_slice = filtered_dataframe[filtered_dataframe["original_index"].isin(development_indices)].copy()
test_slice = filtered_dataframe[filtered_dataframe["original_index"].isin(test_indices)].copy()
load_pool = filtered_dataframe[filtered_dataframe["original_index"].isin(load_pool_indices)].copy()

development_slice_size = len(development_slice)
test_slice_size = len(test_slice)
load_pool_size = len(load_pool)
total_size = development_slice_size + test_slice_size + load_pool_size

# Verify sizes
print(f"""Development slice size = {development_slice_size}
Test slice size = {test_slice_size}
Load pool size = {load_pool_size}
Total size = {total_size}
""")
assert development_slice_size == 500
assert test_slice_size == 1500
assert load_pool_size == 1426
assert total_size == 3426
print("All slice size checks passed!")

# Verify no overlapping indices
assert set(development_indices).isdisjoint(test_indices)
assert set(development_indices).isdisjoint(load_pool_indices)
assert set(test_indices).isdisjoint(load_pool_indices)
print("All overlap checks passed!")

development_slice = pandas.read_csv(os.path.join(folder_path, "data", "development_slice.csv")) 
test_slice = pandas.read_csv( os.path.join(folder_path, "data", "test_slice.csv")) 
load_pool = pandas.read_csv( os.path.join(folder_path, "data", "load_pool_slice.csv"))
print("Original dataframe:", len(original_dataframe))
print("Development:", len(development_slice))
print("Test:", len(test_slice))
print("Load pool:", len(load_pool))
print("Audio column:", "audio" in original_dataframe.columns)

Development slice size = 500
Test slice size = 1500
Load pool size = 1426
Total size = 3426

All slice size checks passed!
All overlap checks passed!


In [21]:
audio_dir = os.path.join(folder_path, "data", "audio")
os.makedirs(audio_dir, exist_ok=True)

selected_indices = numpy.concatenate([
    development_indices,
    test_indices,
    load_pool_indices
])

# Select only the 3,426 recordings
selected_dataframe = original_dataframe.loc[selected_indices]

print("Selected recordings:", len(selected_dataframe))

# 3. Export audio and create manifests
sentiment_rows = []
asr_rows = []

for i, row in selected_dataframe.iterrows():

    audio = row["audio"]

    # Extract audio without TorchCodec
    if audio["bytes"] is not None:
        waveform, sample_rate = soundfile.read(
            io.BytesIO(audio["bytes"])
        )
    else:
        waveform, sample_rate = soundfile.read(audio["path"])

    # Create WAV filename
    filename = os.path.splitext(str(row["file"]))[0] + ".wav"

    audio_path = os.path.abspath(
        os.path.join(audio_dir, filename)
    )

    # Save WAV
    soundfile.write(audio_path, waveform, sample_rate)

    # Sentiment manifest
    sentiment_rows.append({
        "audio_path": audio_path,
        "file": row["file"],
        "label": str(row["label"]).lower()
    })

    # ASR manifest
    asr_rows.append({
        "audio_path": audio_path,
        "text": ""
    })

# 4. Save CSV files
sentiment_df = pandas.DataFrame(sentiment_rows)
asr_df = pandas.DataFrame(asr_rows)

sentiment_df.to_csv(os.path.join(folder_path, "data", "sentiment_audio.csv"), index=False)

asr_df.to_csv(os.path.join(folder_path, "data", "asr_audio.csv"), index=False)

# 5. Verify export
print("\nExport completed!")
print("Audio files:", len(os.listdir(audio_dir)))
print("Sentiment rows:", len(sentiment_df))
print("ASR rows:", len(asr_df))

assert len(sentiment_df) == 3426
assert len(asr_df) == 3426
assert len(os.listdir(audio_dir)) == 3426

print("All checks passed!")


Selected recordings: 3426

Export completed!
Audio files: 3426
Sentiment rows: 3426
ASR rows: 3426
All checks passed!


In [11]:
sentiment_df = pandas.read_csv(os.path.join(folder_path, "data", "sentiment_audio.csv"))
sentiment_df["duration"] = sentiment_df["audio_path"].apply(lambda p: soundfile.info(p).duration)

# attach to the slices by clip name
dur = sentiment_df.set_index("file")["duration"]
for df in (development_slice, test_slice, load_pool):
    df["duration"] = df["file"].map(dur)
    assert df["duration"].notna().all()

# **Whisper (pseudo-gold transcriptions)**

In [24]:
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3", device=device, dtype=torch.float16, chunk_length_s=30, batch_size=4)
print("Whisper loaded successfully!")

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Whisper loaded successfully!


In [30]:
asr_csv_path = os.path.join(folder_path, "data", "asr_audio.csv")
sentiment_csv_path = os.path.join(folder_path, "data", "sentiment_audio.csv")

asr_df = pandas.read_csv(asr_csv_path)
sentiment_df = pandas.read_csv(sentiment_csv_path)

# FIX: Convert transcription columns to string dtype
asr_df["text"] = asr_df["text"].astype("string")

if "transcription" not in sentiment_df.columns:
    sentiment_df["transcription"] = pandas.Series(index=sentiment_df.index, dtype="string")
else:
    sentiment_df["transcription"] = sentiment_df["transcription"].astype("string")

# Verify alignment
assert asr_df["audio_path"].equals(sentiment_df["audio_path"])
assert len(asr_df) == 3426
assert len(sentiment_df) == 3426
assert "text" in asr_df.columns

# Resume previously completed transcriptions
if "transcription" not in sentiment_df.columns:
    sentiment_df["transcription"] = ""

asr_text = asr_df["text"].fillna("").astype(str).str.strip()
sentiment_text = sentiment_df["transcription"].fillna("").astype(str).str.strip()

for i in range(len(asr_df)):
    if asr_text.iloc[i]:
        sentiment_df.loc[i, "transcription"] = asr_text.iloc[i]
    elif sentiment_text.iloc[i]:
        asr_df.loc[i, "text"] = sentiment_text.iloc[i]

# Save progress
def save_progress():
    asr_df.to_csv(asr_csv_path, index=False)
    sentiment_df.to_csv(sentiment_csv_path, index=False)

# Identify remaining recordings
completed_text = asr_df["text"].fillna("").astype(str).str.strip()

pending_indices = [
    i for i, text in enumerate(completed_text)
    if not text
]

print("Total recordings:", len(asr_df))
print("Already completed:", len(asr_df) - len(pending_indices))
print("Remaining:", len(pending_indices))

# Transcribe remaining recordings
batch_size = 4
save_interval = 50

for start in tqdm(range(0, len(pending_indices), batch_size)):

    batch_indices = pending_indices[start:start + batch_size]

    audio_paths = [
        asr_df.loc[i, "audio_path"]
        for i in batch_indices
    ]
    
    try:
        results = []

        for path in audio_paths:
            waveform, sr = soundfile.read(path, dtype="float32")

            result = asr(
                {"raw": waveform, "sampling_rate": sr},
                return_timestamps=False
            )

            results.append(result)

        for i, result in zip(batch_indices, results):
            transcription = result["text"].strip()

            asr_df.loc[i, "text"] = transcription
            sentiment_df.loc[i, "transcription"] = transcription

    except Exception as e:
        print(f"\nBatch error at index {start}: {e}")

    if (start + len(batch_indices)) % save_interval < batch_size:
        save_progress()

# Final save
save_progress()

print("\nTranscription finished!")
print("Total recordings:", len(asr_df))
print("Empty transcriptions:", (asr_df["text"].fillna("").str.strip() == "").sum())

Total recordings: 3426
Already completed: 0
Remaining: 3426


  0%|          | 0/857 [00:00<?, ?it/s]


Transcription finished!
Total recordings: 3426
Empty transcriptions: 0


## Baseline model (always predict "neutral")

In [31]:
def baseline_sentiment_model(dataframe: pandas.DataFrame): # Always predict neutral
    return ["Neutral"] * len(dataframe)

test_slice["baseline_prediction"] = baseline_sentiment_model(test_slice)
print("\nBASELINE MODEL (always predict neutral)\n========================================")
print(classification_report(test_slice["label"], test_slice["baseline_prediction"], zero_division=0))
print()


BASELINE MODEL (always predict neutral)
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        74
     Neutral       0.74      1.00      0.85      1104
    Positive       0.00      0.00      0.00       322

    accuracy                           0.74      1500
   macro avg       0.25      0.33      0.28      1500
weighted avg       0.54      0.74      0.62      1500




## ASR quality

In [13]:
def normalize_text(text):
    text = str(text).lower().replace("—", " ").replace("–", " ")   # dashes between words become spaces
    text = re.sub(r"[^\w\s]", "", text)                             # drops all remaining punctuation, incl. Unicode
    return re.sub(r"\s+", " ", text).strip()

def calculate_wer(df, ref_col, hyp_col):
    ref = df[ref_col].fillna("").map(normalize_text)
    hyp = df[hyp_col].fillna("").map(normalize_text)
    keep = ref != ""                                  # jiwer can't score empty references
    corpus_wer = jiwer.wer(ref[keep].tolist(), hyp[keep].tolist())
    per_clip = pandas.Series(
        [jiwer.wer(r, h) for r, h in zip(ref[keep], hyp[keep])], index=ref[keep].index)
    print(f"{hyp_col}: corpus WER = {corpus_wer:.4f} "
          f"({(~keep).sum()} empty references skipped)")
    return corpus_wer, per_clip

## Gold input ablation
Pseudo-reference/gold transcript:
Use Whisper-large-v3 to transcribe the "true" transcripts.

In [14]:
# 1) Attach pseudo-gold transcripts, WAV paths and durations (no torchcodec, no re-transcribing REF)
REF = "openai/whisper-large-v3"
sent = pandas.read_csv(os.path.join(folder_path, "data", "sentiment_audio.csv")).set_index("file")

for df in (development_slice, test_slice, load_pool):
    df.drop(columns=["audio"], errors="ignore", inplace=True)
    df["audio_path"] = df["file"].map(sent["audio_path"])
    df[f"{REF}_transcript"] = df["file"].map(sent["transcription"]).fillna("")
    df["duration"] = df["audio_path"].map(lambda p: soundfile.info(p).duration)
    assert df["audio_path"].notna().all() and (df["duration"] > 0).all()

# 2) ASR helpers
def get_asr_model(name):
    p = pipeline("automatic-speech-recognition", model=name, device=device)
    kw = {}
    if "whisper" in name:
        kw["chunk_length_s"] = 30
        if getattr(p.model.generation_config, "is_multilingual", False):   # English-only models reject language=
            kw["generate_kwargs"] = {"language": "en", "task": "transcribe"}
    return p, kw

def load_wave(path):
    x, sr = soundfile.read(path, dtype="float32")
    if x.ndim > 1: x = x.mean(axis=1)                                      # force mono
    return {"raw": x, "sampling_rate": sr}

def transcribe(asr, kw, paths, bs=8):
    texts, ms = [], []
    for i in tqdm(range(0, len(paths), bs)):
        batch = [load_wave(p) for p in paths[i:i + bs]]
        t0 = time.perf_counter()
        out = asr(batch, batch_size=bs, **kw)                              # list in -> real batching
        per_clip = (time.perf_counter() - t0) * 1000 / len(batch)
        texts += [o["text"].strip() for o in out]
        ms += [per_clip] * len(batch)
    return texts, ms

# 3) Transcribe every slice once per model, with a cache on disk
ASR_MODELS = ["AventIQ-AI/whisper-audio-to-text", "facebook/s2t-small-librispeech-asr", "openai/whisper-tiny"]
slices = (development_slice, test_slice, load_pool)
all_clips = pandas.concat(slices)[["file", "audio_path"]]
wer_rows = []

for name in ASR_MODELS:
    cache = f"{folder_path}transcripts_{name.split('/')[-1]}.csv"
    if os.path.exists(cache):
        res = pandas.read_csv(cache)
    else:
        asr, kw = get_asr_model(name)
        texts, ms = transcribe(asr, kw, all_clips["audio_path"].tolist())
        res = pandas.DataFrame({"file": all_clips["file"].values, "transcript": texts, "ms": ms})
        res.to_csv(cache, index=False)
        del asr; torch.cuda.empty_cache()
    res = res.set_index("file")
    for df in slices:
        df[f"{name}_transcript"] = df["file"].map(res["transcript"]).fillna("")
        df[f"{name}_transcribe_time_ms"] = df["file"].map(res["ms"])

    corpus_wer, per_clip = calculate_wer(test_slice, f"{REF}_transcript", f"{name}_transcript")
    test_slice[f"{name}_wer"] = per_clip
    wer_rows.append({"asr": name, "wer": corpus_wer})

# 4) Save (no audio column)
pandas.DataFrame(wer_rows).to_csv(f"{folder_path}wer_test.csv", index=False)
development_slice.to_csv(f"{folder_path}development.csv", index=False)
test_slice.to_csv(f"{folder_path}test.csv", index=False)
load_pool.to_csv(f"{folder_path}load_pool.csv", index=False)

C:\Users\nova\AppData\Local\Temp\ipykernel_11396\761102206.py:6: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["audio"], errors="ignore", inplace=True)
C:\Users\nova\AppData\Local\Temp\ipykernel_11396\761102206.py:6: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["audio"], errors="ignore", inplace=True)
C:\Users\nova\AppData\Local\Temp\ipykernel_11396\761102206.py:6: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["audio"], errors="

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

C:\Users\nova\miniconda3\envs\gpu\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nova\.cache\huggingface\hub\models--openai--whisper-tiny. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

... (8 outputs left out)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

  0%|          | 0/429 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (

openai/whisper-tiny_transcript: corpus WER = 0.1988 (0 empty references skipped)


In [21]:
for name in ASR_MODELS:
    cache = f"{folder_path}transcripts_{name.split('/')[-1]}.csv"

    res = pandas.read_csv(cache).set_index("file")

    for df in (development_slice, test_slice, load_pool):
        df[f"{name}_transcript"] = (
            df["file"].map(res["transcript"]).fillna("")
        )
        df[f"{name}_transcribe_time_ms"] = (
            df["file"].map(res["ms"]))

In [22]:
print(development_slice.columns.tolist())

['file', 'instruction', 'label', 'original_index', 'duration', 'audio_path', 'openai/whisper-large-v3_transcript', 'facebook/s2t-small-librispeech-asr_transcript', 'facebook/s2t-small-librispeech-asr_transcribe_time_ms', 'openai/whisper-tiny_transcript', 'openai/whisper-tiny_transcribe_time_ms', 'AventIQ-AI/whisper-audio-to-text_transcript', 'AventIQ-AI/whisper-audio-to-text_transcribe_time_ms']


In [23]:
def tune_confidence_threshold(dataframe, prediction_column, confidence_column, label_column="label"):

    # Actual correctness of sentiment predictions
    correctness = (dataframe[prediction_column] == dataframe[label_column]).astype(int)

    threshold_results = {}

    for threshold in numpy.arange(0.05, 1.00, 0.05):
        threshold = round(threshold, 2)

        confidence_check = (dataframe[confidence_column] >= threshold).astype(int)
        threshold_results[threshold] = balanced_accuracy_score(correctness, confidence_check)

    best_threshold = max(threshold_results, key=threshold_results.get)
    best_balanced_accuracy = threshold_results[best_threshold]
    return best_threshold, best_balanced_accuracy, threshold_results

def get_sentiment_analysis_model(model_name):
    return pipeline("sentiment-analysis", model=model_name, device=device)

def predict_sentiment(sentiment_model, transcription: str):
    start = time.perf_counter()
    result = sentiment_model(transcription)[0]
    end = time.perf_counter()
    sentiment_time_ms = (end - start) * 1000
    return result["label"], result["score"], sentiment_time_ms

# Relabel mapping
def map_siebert_label(label, score, threshold):
    if score < threshold:
        return "Neutral"
    return label.capitalize()

multilingual_label_mapping = {
    "very negative": "Negative",
    "negative": "Negative",
    "neutral": "Neutral",
    "positive": "Positive",
    "very positive": "Positive"
}

# ASSERTIONS
def m1_assertions(y_true, y_pred, confidences, total_seconds):
    checks = {
        "mean_confidence >= 0.70": float(numpy.mean(confidences)) >= 0.70,
        "macro_f1 >= 0.50": f1_score(y_true, y_pred, average="macro") >= 0.50,
        "runtime <= 60 min": total_seconds <= 3600,
    }
    return checks, sum(checks.values())   # e.g. 2 of 3
    
def assert_clips(dataframe: pandas.DataFrame, transcript_column: str="asr_transcription", duration_column: str="duration", confidence_column: str="score", threshold: float=0.7):
    def is_non_empty_transcript(transcript):
        return len(str(transcript).split()) >= 2
    def is_plausible_speech_rate(transcript, duration):
        word_count = len(str(transcript).split())
        if duration <= 0:
            return False
        speech_rate = word_count / duration
        return 0.5 <= speech_rate <= 5.0
    def is_confident_classification(confidence):
        return confidence >= threshold
    dataframe["assert_non_empty_transcript"] = dataframe[transcript_column].apply(is_non_empty_transcript)
    dataframe["assert_plausible_speech_rate"] = dataframe.apply(lambda row:
                                                                is_plausible_speech_rate(row[transcript_column], row[duration_column]),
                                                                axis=1)
    dataframe["assert_confident_classification"] = dataframe[confidence_column].apply(is_confident_classification)
    dataframe["assert_passed"] = dataframe[["assert_non_empty_transcript", "assert_plausible_speech_rate", "assert_confident_classification"]].all(axis=1)
    return dataframe

def map_labels(model, raw, score, tau=None):
    if "cardiff" in model:      return raw.str.capitalize()
    if "multilingual" in model: return raw.str.lower().map(multilingual_label_mapping)
    return pandas.Series([map_siebert_label(l, s, tau) for l, s in zip(raw, score)], index=raw.index)

def run_clf(clf, texts):
    idx = [i for i, t in zip(texts.index, texts) if str(t).strip()]
    outs = clf([texts[i] for i in idx], truncation=True, max_length=512, batch_size=32)
    lab = pandas.Series("neutral", index=texts.index); sc = pandas.Series(0.0, index=texts.index)
    for i, o in zip(idx, outs): lab[i], sc[i] = o["label"], o["score"]
    return lab, sc

SENTIMENT_MODELS = [
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "tabularisai/multilingual-sentiment-analysis",
    "siebert/sentiment-roberta-large-english", 
    "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
]

REF = "openai/whisper-large-v3"
results, thresholds, gold_cache = [], {}, {}
assert set(test_slice["label"]) == {"Negative", "Neutral", "Positive"}, test_slice["label"].unique()

for s_name in SENTIMENT_MODELS:
    clf = get_sentiment_analysis_model(s_name)
    for a_name in [m for m in ASR_MODELS if m != REF]:
        cfg = f"{a_name}|{s_name}"
        for src, tcol in [("full", f"{a_name}_transcript"), ("gold", f"{REF}_transcript")]:
            if src == "gold" and s_name in gold_cache:   # 2) first lines inside the inner loop
                results.append({**gold_cache[s_name], "config": cfg})
                continue
            dev_raw, dev_sc = run_clf(clf, development_slice[tcol])
            t0 = time.perf_counter()
            tst_raw, tst_sc = run_clf(clf, test_slice[tcol])
            sent_s = time.perf_counter() - t0
            tau = None
            if "siebert" in s_name or "distilbert" in s_name:   # tune tau on dev macro-F1
                grid = [round(x, 2) for x in numpy.arange(0.50, 1.00, 0.01)]
                tau = max(grid, key=lambda g: f1_score(development_slice["label"],
                          map_labels(s_name, dev_raw, dev_sc, g), average="macro"))
            dev_pred = map_labels(s_name, dev_raw, dev_sc, tau)
            tst_pred = map_labels(s_name, tst_raw, tst_sc, tau)
            y, yd = test_slice["label"], development_slice["label"]

            row = dict(config=cfg, input=src, tau=tau,
                       macro_f1=f1_score(y, tst_pred, average="macro"),
                       accuracy=accuracy_score(y, tst_pred))
            if src == "full":
                # tune t on dev (common rule)
                correct = (dev_pred == yd).astype(int)
                grid = [round(x, 2) for x in numpy.arange(0.05, 1.00, 0.05)]
                t = max(grid, key=lambda g: balanced_accuracy_score(correct, (dev_sc >= g).astype(int)))
                thresholds[cfg] = {"t": float(t), "tau": None if tau is None else float(tau)}
                tr = test_slice[tcol].fillna("")
                nwords = tr.str.split().str.len()
                a1 = nwords >= 2
                a2 = (nwords / test_slice["duration"]).between(0.5, 5.0)
                a3 = tst_sc >= t
                ok = a1 & a2 & a3
                correct_test = (tst_pred == y)
                row.update(acc_pass=correct_test[ok].mean(), acc_fail=correct_test[~ok].mean())
                row.update(t=t, A1=a1.mean(), A2=a2.mean(), A3=a3.mean(),
                           assert_pass=(a1 & a2 & a3).mean())
                rep = classification_report(y, tst_pred, zero_division=0, output_dict=True)
                name = f"{a_name.split('/')[-1]}__{s_name.split('/')[-1]}"
                pandas.DataFrame(rep).T.to_csv(f"{folder_path}report_{name}.csv")
                
                asr_s = test_slice[f"{a_name}_transcribe_time_ms"].sum() / 1000
                total_s = asr_s + sent_s
                checks, passed = m1_assertions(y, tst_pred, tst_sc, total_s)
                row.update(mean_conf=tst_sc.mean(), asr_s=asr_s, sent_s=sent_s,
                           total_s=total_s, m1_passed=passed)
                
                test_slice[f"{cfg}_score"] = tst_sc     # per-clip confidence, needed for A3
                test_slice[f"{cfg}_pred"] = tst_pred
            results.append(row)
            if src == "gold":
                gold_cache[s_name] = row

res = pandas.DataFrame(results)
res.to_csv(f"{folder_path}m2_t1_results.csv", index=False)
test_slice.to_csv(f"{folder_path}test_scored.csv", index=False)
json.dump(thresholds, open(f"{folder_path}thresholds.json", "w"), indent=2)

# Agreement between assertion pass rate and true accuracy
full = res[res.input == "full"]
print("Pearson:", full["assert_pass"].corr(full["macro_f1"]),
      "Spearman:", full["assert_pass"].corr(full["macro_f1"], method="spearman"))
gold = res[res.input == "gold"].set_index("config")["macro_f1"]
full = full.set_index("config"); full["upstream_loss"] = gold - full["macro_f1"]
full[["macro_f1", "accuracy", "assert_pass", "upstream_loss"]].to_csv(f"{folder_path}m2_t1_summary.csv")
print(full[["macro_f1", "assert_pass", "upstream_loss"]])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

C:\Users\nova\miniconda3\envs\gpu\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nova\.cache\huggingface\hub\models--tabularisai--multilingual-sentiment-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Pearson: -0.2511710532247963 Spearman: -0.16666666666666669
                                                    macro_f1  assert_pass  \
config                                                                      
AventIQ-AI/whisper-audio-to-text|cardiffnlp/twi...  0.621092     0.690667   
facebook/s2t-small-librispeech-asr|cardiffnlp/t...  0.610226     0.503333   
openai/whisper-tiny|cardiffnlp/twitter-roberta-...  0.614975     0.584000   
AventIQ-AI/whisper-audio-to-text|tabularisai/mu...  0.330412     0.679333   
facebook/s2t-small-librispeech-asr|tabularisai/...  0.337207     0.447333   
openai/whisper-tiny|tabularisai/multilingual-se...  0.325207     0.332000   
AventIQ-AI/whisper-audio-to-text|siebert/sentim...  0.334118     0.985333   
facebook/s2t-small-librispeech-asr|siebert/sent...  0.336843     0.833333   
openai/whisper-tiny|siebert/sentiment-roberta-l...  0.322849     0.982667   

                                                    upstream_loss  
config                  

In [34]:
def map_siebert_label(label, score, threshold):
    if isinstance(score, pandas.Series):
        return pandas.Series(
            [
                "Neutral" if s < threshold else str(l).capitalize()
                for l, s in zip(label, score)
            ],
            index=label.index
        )

    if score < threshold:
        return "Neutral"

    return str(label).capitalize()
    
CHOICE_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
s_name = CHOICE_MODEL
clf = get_sentiment_analysis_model(s_name)

new_results = []
new_thresholds = {}

for a_name in [m for m in ASR_MODELS if m != REF]:

    cfg = f"{a_name}|{s_name}"

    for src, tcol in [
        ("full", f"{a_name}_transcript"),
        ("gold", f"{REF}_transcript")
    ]:

        dev_raw, dev_sc = run_clf(clf, development_slice[tcol])

        t0 = time.perf_counter()
        tst_raw, tst_sc = run_clf(clf, test_slice[tcol])
        sent_s = time.perf_counter() - t0

        # Tune Neutral threshold on development data
        grid_tau = [round(x, 2) for x in numpy.arange(0.50, 1.00, 0.01)]

        tau = max(
            grid_tau,
            key=lambda g: f1_score(
                development_slice["label"],
                map_siebert_label(dev_raw, dev_sc, g),
                average="macro"
            )
        )

        dev_pred = pandas.Series(
            [map_siebert_label(l, s, tau)
             for l, s in zip(dev_raw, dev_sc)],
            index=dev_raw.index
        )

        tst_pred = pandas.Series(
            [map_siebert_label(l, s, tau)
             for l, s in zip(tst_raw, tst_sc)],
            index=tst_raw.index
        )

        y, yd = test_slice["label"], development_slice["label"]

        row = dict(
            config=cfg,
            input=src,
            tau=tau,
            macro_f1=f1_score(y, tst_pred, average="macro"),
            accuracy=accuracy_score(y, tst_pred)
        )

        if src == "full":

            correct = (dev_pred == yd).astype(int)
            grid = [round(x, 2) for x in numpy.arange(0.05, 1.00, 0.05)]

            t = max(
                grid,
                key=lambda g: balanced_accuracy_score(
                    correct, (dev_sc >= g).astype(int)
                )
            )

            new_thresholds[cfg] = {
                "t": float(t),
                "tau": float(tau)
            }

            tr = test_slice[tcol].fillna("")
            nwords = tr.str.split().str.len()

            a1 = nwords >= 2
            a2 = (nwords / test_slice["duration"]).between(0.5, 5.0)
            a3 = tst_sc >= t

            ok = a1 & a2 & a3
            correct_test = tst_pred == y

            row.update(
                acc_pass=correct_test[ok].mean(),
                acc_fail=correct_test[~ok].mean(),
                t=t,
                A1=a1.mean(),
                A2=a2.mean(),
                A3=a3.mean(),
                assert_pass=ok.mean()
            )

            asr_s = test_slice[f"{a_name}_transcribe_time_ms"].sum() / 1000
            total_s = asr_s + sent_s

            checks, passed = m1_assertions(y, tst_pred, tst_sc, total_s)

            row.update(
                mean_conf=tst_sc.mean(),
                asr_s=asr_s,
                sent_s=sent_s,
                total_s=total_s,
                m1_passed=passed
            )

            test_slice[f"{cfg}_score"] = tst_sc
            test_slice[f"{cfg}_pred"] = tst_pred

        new_results.append(row)

        # Save classification report
        name = f"{a_name.split('/')[-1]}__{s_name.split('/')[-1]}"
        pandas.DataFrame(
            classification_report(
                y, tst_pred, zero_division=0, output_dict=True
            )
        ).T.to_csv(f"{folder_path}report_{name}.csv")


# Append to existing results WITHOUT overwriting other models
old_results = pandas.read_csv(
    f"{folder_path}m2_t1_results.csv"
)

updated_results = pandas.concat(
    [old_results, pandas.DataFrame(new_results)],
    ignore_index=True
).drop_duplicates(
    subset=["config", "input"],
    keep="last"
)

updated_results.to_csv(
    f"{folder_path}m2_t1_results.csv",
    index=False
)

test_slice.to_csv(
    f"{folder_path}test_scored.csv",
    index=False
)

# Append thresholds without losing existing models
threshold_path = f"{folder_path}thresholds.json"

if os.path.exists(threshold_path):
    with open(threshold_path, "r") as f:
        old_thresholds = json.load(f)
else:
    old_thresholds = {}

old_thresholds.update(new_thresholds)

with open(threshold_path, "w") as f:
    json.dump(old_thresholds, f, indent=2)

print(pandas.DataFrame(new_results)[
    ["config", "input", "tau", "macro_f1", "accuracy"]
])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

                                              config input   tau  macro_f1  \
0  AventIQ-AI/whisper-audio-to-text|distilbert/di...  full  0.99  0.458020   
1  AventIQ-AI/whisper-audio-to-text|distilbert/di...  gold  0.99  0.465139   
2  facebook/s2t-small-librispeech-asr|distilbert/...  full  0.99  0.447990   
3  facebook/s2t-small-librispeech-asr|distilbert/...  gold  0.99  0.465139   
4  openai/whisper-tiny|distilbert/distilbert-base...  full  0.99  0.457491   
5  openai/whisper-tiny|distilbert/distilbert-base...  gold  0.99  0.465139   

   accuracy  
0  0.519333  
1  0.530000  
2  0.525333  
3  0.530000  
4  0.524000  
5  0.530000  


In [35]:
old_results = pandas.read_csv(f"{folder_path}m2_t1_results.csv")

updated_results = pandas.concat(
    [old_results, pandas.DataFrame(new_results)],
    ignore_index=True
)

updated_results = updated_results.drop_duplicates(
    subset=["config", "input"],
    keep="last"
)

updated_results.to_csv(
    f"{folder_path}m2_t1_results.csv",
    index=False
)

test_slice.to_csv(f"{folder_path}test_scored.csv", index=False)

print(updated_results.tail(len(new_results)))

                                               config input   tau  macro_f1  \
24  AventIQ-AI/whisper-audio-to-text|distilbert/di...  full  0.99  0.458020   
25  AventIQ-AI/whisper-audio-to-text|distilbert/di...  gold  0.99  0.465139   
26  facebook/s2t-small-librispeech-asr|distilbert/...  full  0.99  0.447990   
27  facebook/s2t-small-librispeech-asr|distilbert/...  gold  0.99  0.465139   
28  openai/whisper-tiny|distilbert/distilbert-base...  full  0.99  0.457491   
29  openai/whisper-tiny|distilbert/distilbert-base...  gold  0.99  0.465139   

    accuracy  acc_pass  acc_fail     t   A1        A2   A3  assert_pass  \
24  0.519333  0.518945  0.545455  0.05  1.0  0.985333  1.0     0.985333   
25  0.530000       NaN       NaN   NaN  NaN       NaN  NaN          NaN   
26  0.525333  0.530400  0.500000  0.05  1.0  0.833333  1.0     0.833333   
27  0.530000       NaN       NaN   NaN  NaN       NaN  NaN          NaN   
28  0.524000  0.527815  0.307692  0.05  1.0  0.982667  1.0     0.982667

## Request & SLA unit

In [30]:
def calculate_latency_metrics(dataframe, processing_column):

    processing_time = dataframe[processing_column]

    duration = dataframe["duration"]

    rtf = (processing_time / 1000) / duration

    return {
        "total_time_s": test_slice[processing_column].sum() / 1000,
        "mean_latency_ms": processing_time.mean(),
        "median_latency_ms": processing_time.median(),
        "p95_latency_ms": processing_time.quantile(0.95),
        "mean_rtf": rtf.mean(),
        "median_rtf": rtf.median(),
        "p95_rtf": rtf.quantile(0.95)
    }

In [31]:
latency_rows = []
for name in ASR_MODELS:
    processing_column = f"{name}_transcribe_time_ms"
    if processing_column not in test_slice.columns:
        print(f"Skipping {name}: latency column missing")
        continue

    metrics = calculate_latency_metrics(test_slice,processing_column)

    latency_rows.append({"asr": name, **metrics})

latency_df = pandas.DataFrame(latency_rows)

latency_df.to_csv(f"{folder_path}latency_test.csv", index=False)

print(latency_df.to_string(index=False))

                               asr  total_time_s  mean_latency_ms  median_latency_ms  p95_latency_ms  mean_rtf  median_rtf  p95_rtf
  AventIQ-AI/whisper-audio-to-text    126.363616        84.242411          80.313538      144.941437  0.012857    0.011196 0.026000
facebook/s2t-small-librispeech-asr    252.813044       168.542029         177.575813      249.576162  0.026041    0.024802 0.049142
               openai/whisper-tiny    142.350400        94.900267          76.734000      162.974675  0.014574    0.011509 0.029003
